# `toxpol.datagen` — Synthetic Data Generation Demo

Polarized Trees needs datasets where the ground truth is *known* — which demographic dimensions drive disagreement and how. Real annotation data can't provide this, so we generate synthetic datasets with **injected, controlled polarization**.

This notebook demonstrates that `AnnotatorPool` produces datasets where:
- injected polarization is clearly visible in the rating distributions
- the bias config (ground truth) is returned alongside the data, enabling direct recovery evaluation

**Requires:** `pip install "toxpol-nlp[ndfu]"`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ndfu import dfu

from toxpol.datagen import AnnotatorPool, DEFAULT_DIMENSIONS

## 1. Build the annotator pool

`DEFAULT_DIMENSIONS` covers gender, politics, age, education, and orientation.
Pass a custom dict to use different dimensions or values.
Use `exclude` to drop dimensions for ablation runs.

In [ ]:
pool = AnnotatorPool(
    DEFAULT_DIMENSIONS,
    # exclude=["education"],        # drop a dimension
    annotators_per_identity=10,     # replication factor per unique identity
    scale=5,
    toxic_range=(4, 5),
    civil_range=(1, 2),
    neutral_range=(3, 3),
)
pool.summary()

## 2. Generate a dataset

One bias config is drawn for the whole call — all texts share the same
polarization structure. Call `generate_dataset()` again for a fresh config.

In [ ]:
dataset, bias_config = pool.generate_dataset(
    n_texts=10,
    n_annotators_per_text=100,  # must be <= pool.pool_size (1620)
    noise=0.1,                  # prob. of fully random rating (outlier noise)
    polarizing_prob=0.7,        # prob. each dimension is polarizing vs. unimodal
)

print(dataset.head(10))
print("\nBias config:")
for dim, config in bias_config.items():
    print(f"  {dim}: {config}")

## 3. Compute nDFU per dimension

nDFU (normalized Disagreement from Uniformity) measures how bimodal a rating
distribution is. Higher = more polarized. We compute it overall and per
dimension value, then compare against the known bias config.

In [ ]:
pool.describe_bias(bias_config)


## 4. Visualise — rating distributions by dimension value

Polarizing dimensions should show clearly separated distributions between
toxic-pole and civil-pole values.

In [ ]:
polarizing_dims = [d for d, c in bias_config.items() if c["role"] == "polarizing"]

text_data = dataset[dataset["text_id"] == 0]
scale = pool.scale
bins = range(1, scale + 2)

for dim in polarizing_dims:
    toxic_pole = bias_config[dim]["toxic_pole"]
    civil_pole = bias_config[dim]["civil_pole"]

    fig, axes = plt.subplots(1, len(bias_config[dim]["toxic_pole"]) + len(civil_pole),
                             figsize=(3 * (len(toxic_pole) + len(civil_pole)), 3),
                             sharey=True)
    fig.suptitle(f"{dim}  [toxic: {toxic_pole}  |  civil: {civil_pole}]")

    all_values = toxic_pole + civil_pole
    for ax, value in zip(axes, all_values):
        subset = text_data[text_data[dim] == value]["rating"]
        color = "tomato" if value in toxic_pole else "steelblue"
        ax.hist(subset, bins=bins, align="left", color=color, edgecolor="white")
        ax.set_title(value)
        ax.set_xlabel("rating")
        ax.set_xticks(range(1, scale + 1))

    axes[0].set_ylabel("count")
    plt.tight_layout()
    plt.show()